In [ ]:
import warnings
warnings.filterwarnings('ignore')

# ✅ AFTER RUNNING THIS CELL:
# All Python warnings (deprecation, convergence, etc.) are suppressed.
# You won't see any yellow warning messages cluttering the output during the project.


In [ ]:
import micropip
await micropip.install("imbalanced-learn")

# ✅ AFTER RUNNING THIS CELL:
# The 'imbalanced-learn' library is installed in your environment.
# This library provides SMOTE (Synthetic Minority Over-sampling Technique)
# which we'll use later to fix class imbalance in the dataset.


In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from skimage.feature import local_binary_pattern

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, auc, ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

from imblearn.over_sampling import SMOTE

# ✅ AFTER RUNNING THIS CELL:
# All required libraries are imported and ready to use:
#   - cv2 (OpenCV)     → for image reading, resizing, edge detection
#   - numpy            → for numerical array operations
#   - matplotlib       → for plotting graphs and images
#   - skimage (LBP)    → for texture feature extraction
#   - sklearn tools    → for ML models, scaling, PCA, evaluation metrics
#   - SMOTE            → for handling class imbalance
# If any import fails here, install the missing library and re-run.


In [ ]:
# CHANGE THIS PATH TO YOUR DATASET FOLDER
BASE_PATH = "cv_dataset/train"

# auto detects class folders inside BASE_PATH — no need to change
class_folders = sorted([d for d in os.listdir(BASE_PATH) if os.path.isdir(os.path.join(BASE_PATH, d))])
class_map = {cls: idx for idx, cls in enumerate(class_folders)}

# auto picks first valid image from each class for demos
demo_images = {}
for cls in class_folders:
    folder = os.path.join(BASE_PATH, cls)
    for fname in os.listdir(folder):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            demo_images[cls] = os.path.join(folder, fname)
            break

first_demo_path = list(demo_images.values())[0]

print("Classes found:", class_map)
print("Demo images:", demo_images)

# ✅ AFTER RUNNING THIS CELL:
# The dataset folder is scanned automatically.
# You will see printed output like:
#   Classes found: {'cat': 0, 'dog': 1, ...}   ← class name → numeric label mapping
#   Demo images: {'cat': 'cv_dataset/train/cat/img1.jpg', ...}  ← one sample per class
# These variables (class_map, demo_images) are used throughout the rest of the notebook.
# ⚠ If you see a FileNotFoundError here, check that BASE_PATH is correct.


In [ ]:
def preprocess_image(img):
    img = cv2.resize(img, (128, 128))
    # blur before colour conversion to preserve colour integrity
    img = cv2.medianBlur(img, 5)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)
    return img, hsv, gray

# ✅ AFTER RUNNING THIS CELL:
# The preprocess_image() function is defined (not yet called).
# When called on any image it will:
#   1. Resize it to 128×128 pixels (uniform input size)
#   2. Apply median blur to reduce noise
#   3. Convert to HSV color space (good for color-based features)
#   4. Convert to Grayscale + apply histogram equalization (enhances contrast)
# Returns: (processed BGR image, HSV image, equalized grayscale image)


In [ ]:
def augment_image(img):
    augmented = [img]
    augmented.append(cv2.flip(img, 1))
    augmented.append(np.clip(img.astype(np.int32) + 30, 0, 255).astype(np.uint8))
    augmented.append(np.clip(img.astype(np.int32) - 30, 0, 255).astype(np.uint8))
    noise = np.random.normal(0, 10, img.shape).astype(np.int32)
    augmented.append(np.clip(img.astype(np.int32) + noise, 0, 255).astype(np.uint8))
    return augmented

# ✅ AFTER RUNNING THIS CELL:
# The augment_image() function is defined (not yet called).
# When called on any image it produces 5 versions:
#   1. Original image (unchanged)
#   2. Horizontally flipped (mirror)
#   3. Brighter (+30 brightness)
#   4. Darker  (-30 brightness)
#   5. Gaussian noise added
# Purpose: artificially expands the dataset so the model generalises better.


In [ ]:
def show_sample_images(base_path, class_folders, samples=4):
    plt.figure(figsize=(3*samples, 3*len(class_folders)))
    idx = 1
    for cls in class_folders:
        folder = os.path.join(base_path, cls)
        images = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))][:samples]
        for img_name in images:
            img = cv2.imread(os.path.join(folder, img_name))
            if img is None:
                continue
            plt.subplot(len(class_folders), samples, idx)
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            plt.title(cls.capitalize())
            plt.axis("off")
            idx += 1
    plt.suptitle("Sample Images from Dataset", fontsize=14)
    plt.tight_layout()
    plt.show()

show_sample_images(BASE_PATH, class_folders, samples=4)

# ✅ AFTER RUNNING THIS CELL:
# A grid of sample images from your dataset is displayed.
# Each row = one class, each column = one sample image.
# This is a visual sanity check — confirm that:
#   - Images are loading correctly
#   - Class folders contain the right images
#   - Labels look correct


In [ ]:
def show_preprocessing_demo(img_path):
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img, hsv, gray = preprocess_image(img)
    edges = cv2.Canny(gray, 100, 200)

    plt.figure(figsize=(10, 6))
    plt.subplot(1, 4, 1); plt.imshow(img_rgb); plt.title("Original"); plt.axis("off")
    plt.subplot(1, 4, 2); plt.imshow(gray, cmap="gray"); plt.title("Grayscale + Equalized"); plt.axis("off")
    plt.subplot(1, 4, 3); plt.imshow(hsv[:,:,0], cmap="hsv"); plt.title("HSV (Hue Channel)"); plt.axis("off")
    plt.subplot(1, 4, 4); plt.imshow(edges, cmap="gray"); plt.title("Edge Detection (Canny)"); plt.axis("off")
    plt.tight_layout()
    plt.show()

for cls, path in demo_images.items():
    show_preprocessing_demo(path)

# ✅ AFTER RUNNING THIS CELL:
# For each class, a 4-panel figure is shown with:
#   Panel 1: Original image (raw from disk)
#   Panel 2: Grayscale + Histogram Equalized (enhanced contrast)
#   Panel 3: HSV Hue Channel (color info separated from brightness)
#   Panel 4: Canny Edge Detection (outlines/boundaries of the object)
# This lets you visually verify that each preprocessing step is working correctly.


In [ ]:
def show_augmentation_demo(img_path):
    img_orig = cv2.imread(img_path)
    augs = augment_image(cv2.resize(img_orig, (128, 128)))
    labels = ["Original", "H-Flip", "Brighter", "Darker", "Gaussian Noise"]
    plt.figure(figsize=(16, 3.5))
    for i, (aug, lbl) in enumerate(zip(augs, labels)):
        plt.subplot(1, 5, i+1)
        plt.imshow(cv2.cvtColor(aug, cv2.COLOR_BGR2RGB))
        plt.title(lbl)
        plt.axis('off')
    plt.suptitle("Data Augmentation Techniques", fontsize=14)
    plt.tight_layout()
    plt.show()

show_augmentation_demo(first_demo_path)

# ✅ AFTER RUNNING THIS CELL:
# A side-by-side comparison of all 5 augmented versions of one sample image is shown:
#   Original | H-Flip | Brighter | Darker | Gaussian Noise
# This confirms augmentation is working and shows how each technique transforms the image.
# All 5 versions of every image will be used in model training.


In [ ]:
def extract_features(img, hsv, gray):
    features = []

    # colour feature — HSV hue histogram 32 bins
    hist = cv2.calcHist([hsv], [0], None, [32], [0, 180])
    hist = hist.astype("float32")
    hist /= (hist.sum() + 1e-6)
    features.extend(hist.flatten())

    # texture feature — LBP histogram
    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0,10))
    lbp_hist = lbp_hist.astype("float32")
    lbp_hist /= (lbp_hist.sum() + 1e-6)
    features.extend(lbp_hist)

    # shape feature — contour area, perimeter, circularity
    edges = cv2.Canny(gray, 100, 200)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    area = sum(cv2.contourArea(c) for c in contours)
    perimeter = sum(cv2.arcLength(c, True) for c in contours)
    area = np.clip(area, 0, 1e5)
    perimeter = np.clip(perimeter, 0, 1e4)
    circularity = (4 * np.pi * area) / (perimeter**2 + 1e-6)
    features.extend([area, perimeter, circularity])

    return np.array(features, dtype=np.float32)

# ✅ AFTER RUNNING THIS CELL:
# The extract_features() function is defined (not yet called).
# When called, it computes a 45-dimensional feature vector from one image:
#   - 32 values : HSV Hue Histogram  → captures color distribution
#   - 10 values : LBP Histogram      → captures texture patterns
#   -  3 values : Contour features   → area, perimeter, circularity (shape)
# This vector is what the ML classifiers will learn from.


In [ ]:
def show_contours(img_path):
    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 100, 200)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contour_img = img.copy()
    cv2.drawContours(contour_img, contours, -1, (0, 255, 0), 2)
    plt.figure(figsize=(4, 4))
    plt.imshow(cv2.cvtColor(contour_img, cv2.COLOR_BGR2RGB))
    plt.title("Contour-Based Shape Features")
    plt.axis("off")
    plt.show()

for cls, path in demo_images.items():
    show_contours(path)

# ✅ AFTER RUNNING THIS CELL:
# For each class, an image is shown with green contour lines drawn over it.
# These contours are the object boundaries detected by Canny edge detection.
# The contour area, perimeter, and circularity extracted from these lines
# form the 'shape features' used in classification.


In [ ]:
img = cv2.imread(first_demo_path)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
sift = cv2.SIFT_create()
keypoints, descriptors = sift.detectAndCompute(gray, None)
sift_img = cv2.drawKeypoints(img, keypoints, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

# ✅ AFTER RUNNING THIS CELL:
# SIFT (Scale-Invariant Feature Transform) keypoints and descriptors
# are computed for the first demo image and stored in:
#   keypoints   → list of detected interest points
#   descriptors → 128-dimensional descriptor for each keypoint
#   sift_img    → the image with keypoints drawn on it (used in next cell)
# Note: SIFT is computed here for visualisation only; it is NOT used as a feature in training.


In [ ]:
plt.subplot(3,4,12); plt.imshow(sift_img); plt.title("SIFT"); plt.axis('off')

# ✅ AFTER RUNNING THIS CELL:
# A subplot showing the SIFT keypoints visualisation is added to a figure.
# Each circle represents a detected keypoint — its size shows the scale,
# and the line inside shows the orientation.
# This is a standalone demo of what SIFT detects; it is not part of the training pipeline.


In [ ]:
def load_dataset(base_path, class_map):
    X, y = [], []

    for cls, label in class_map.items():
        folder = os.path.join(base_path, cls)
        num = 0

        for file in os.listdir(folder):
            img_path = os.path.join(folder, file)
            try:
                img = cv2.imread(img_path)

                if img is None:
                    print("Corrupt Image (None):", file)
                    continue

                if img.size == 0:
                    print("Corrupt Image (empty):", file)
                    continue

                if len(img.shape) < 3 or img.shape[2] != 3:
                    print("Corrupt Image (bad shape):", file)
                    continue

                candidates = augment_image(img)

                for candidate in candidates:
                    img_p, hsv, gray = preprocess_image(candidate)
                    features = extract_features(img_p, hsv, gray)
                    X.append(features)
                    y.append(label)

                num += 1
                print("Number of images Done", num, end="\r")

            except Exception as e:
                print("Corrupt Image", e)
                continue

        print(f"\nLabel {label} Done")

    return np.array(X), np.array(y)

# ✅ AFTER RUNNING THIS CELL:
# The load_dataset() function is defined (not yet called).
# When called, it will:
#   1. Loop through every image in every class folder
#   2. Skip corrupt or invalid images (with a printed warning)
#   3. Apply augmentation (×5 images per original)
#   4. Preprocess each image and extract its feature vector
#   5. Return X (feature matrix) and y (labels array)
# Progress is printed as 'Number of images Done N' while it runs.


In [ ]:
X, y = load_dataset(BASE_PATH, class_map)

# remove any NaN or Inf values produced by corrupt images
mask = np.isfinite(X).all(axis=1)
X = X[mask]
y = y[mask]

print(f"Dataset loaded: {X.shape[0]} samples, {X.shape[1]} features")
for cls, label in class_map.items():
    print(f"  {cls}: {(y==label).sum()} samples")

# ✅ AFTER RUNNING THIS CELL (takes the longest — be patient ⏳):
# The entire dataset is loaded, augmented, and feature-extracted.
# You will see printed output like:
#   Number of images Done 50   (progress per class)
#   Label 0 Done
#   Label 1 Done ...
#   Dataset loaded: 500 samples, 45 features
#   cat: 100 samples | dog: 100 samples ...
# X → shape (N_samples, 45) feature matrix
# y → shape (N_samples,)    label array
# Any NaN/Inf values from corrupt images are automatically removed.


In [ ]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

print(f"After SMOTE — Total: {len(y_resampled)}")
for cls, label in class_map.items():
    print(f"  {cls}: {(y_resampled==label).sum()}")
print(f"Train: {len(y_train)} | Test: {len(y_test)}")

# ✅ AFTER RUNNING THIS CELL:
# Class imbalance is fixed using SMOTE (synthetic oversampling).
# You will see:
#   After SMOTE — Total: N  (all classes now have equal samples)
#   cat: 200 | dog: 200 ...  (each class balanced to the majority count)
#   Train: 400 | Test: 100
# Then the balanced data is split 80/20 into:
#   X_train, y_train → used to train the models
#   X_test,  y_test  → held out, used only for final evaluation
# stratify=y ensures each class is proportionally represented in both splits.


In [ ]:
k_values = [3, 4, 5, 6, 7, 8, 9]

# ✅ AFTER RUNNING THIS CELL:
# A list of k values [3, 4, 5, 6, 7, 8, 9] is stored in memory.
# These are the candidate neighbor counts that will be compared
# in the k-NN learning curve plot in the next cell.
# No output is printed — this cell just defines the list.


In [ ]:
plt.figure(figsize=(16, 8))

for k in k_values:
    knn_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k, weights="distance", metric="minkowski"))
    ])

    train_sizes, train_scores, val_scores = learning_curve(
        estimator=knn_pipeline,
        X=X_resampled,
        y=y_resampled,
        train_sizes=np.linspace(0.1, 1.0, 6),
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )

    val_mean = np.mean(val_scores, axis=1)
    plt.plot(train_sizes, val_mean, marker="o", label=f"k = {k}")

plt.xlabel("Training Set Size")
plt.ylabel("Validation Accuracy")
plt.title("Learning Curves of k-NN for Different k Values")
plt.legend()
plt.grid(True)
plt.show()

# ✅ AFTER RUNNING THIS CELL:
# A line chart is displayed showing validation accuracy vs training set size
# for each value of k (3 through 9).
# How to read it:
#   - Lines that flatten quickly → the model is learning efficiently
#   - The k with the highest and most stable curve → best choice
# Use this plot to visually confirm that k=4 (or whichever k you choose) is optimal.
# This step takes a few minutes as it runs 5-fold CV for each k.


In [ ]:
models = {
    "KNN (k=4)":           KNeighborsClassifier(n_neighbors=4, weights='distance', metric='minkowski'),
    "SVM (RBF)":           SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    "Decision Tree":       DecisionTreeClassifier(max_depth=10, random_state=42),
    "Naive Bayes":         GaussianNB(),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

print(f"{'Model':<24} {'Mean CV Acc':>12} {'Std':>8}")
print("-" * 46)

for name, clf in models.items():
    pipeline = Pipeline([('scaler', StandardScaler()), ('clf', clf)])
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    cv_results[name] = scores
    print(f"{name:<24} {scores.mean():.4f}       +/-{scores.std():.4f}")

# ✅ AFTER RUNNING THIS CELL:
# All 7 classifiers are defined and evaluated using 5-Fold Cross Validation.
# A table is printed:
#   Model                    Mean CV Acc     Std
#   KNN (k=4)                0.XXXX       +/-0.XX
#   SVM (RBF)                0.XXXX       +/-0.XX  ... etc
# Each model is tested inside a Pipeline: StandardScaler → Classifier.
# The best_name variable is NOT set yet — it is set in the next cell.
# ⚠ This step may take several minutes depending on dataset size.


In [ ]:
names  = list(cv_results.keys())
means  = [cv_results[n].mean() for n in names]
stds   = [cv_results[n].std()  for n in names]
colors = ['#e74c3c' if m == max(means) else '#3498db' for m in means]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(names, means, xerr=stds, color=colors, edgecolor='white', height=0.55, capsize=4)
ax.set_xlabel('Mean CV Accuracy', fontsize=12)
ax.set_title('5-Fold CV Model Comparison', fontsize=14)
ax.set_xlim(0.5, 1.02)
for bar, val in zip(bars, means):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=10)
ax.legend(handles=[mpatches.Patch(color='#e74c3c', label='Best'), mpatches.Patch(color='#3498db', label='Others')], loc='lower right')
plt.tight_layout()
plt.show()

best_name = names[np.argmax(means)]
print(f"Best model: {best_name} (CV Accuracy = {max(means):.4f})")

# ✅ AFTER RUNNING THIS CELL:
# A horizontal bar chart is displayed comparing all 7 models by mean CV accuracy.
#   - The best model bar is highlighted in RED
#   - All other bars are in BLUE
#   - Error bars show the standard deviation across 5 folds
# The best model name is printed:
#   Best model: <ModelName> (CV Accuracy = 0.XXXX)
# The variable 'best_name' is now set and used in all subsequent cells.


In [ ]:
# train knn k=4 first exactly like teacher notebook
k = 4
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=k, weights="distance", metric="minkowski"))
])
knn_pipeline.fit(X_train, y_train)
y_pred = knn_pipeline.predict(X_test)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=list(class_map.keys())))

# ✅ AFTER RUNNING THIS CELL:
# k-NN (k=4) is trained on X_train and evaluated on X_test.
# Two results are printed:
#   1. Confusion Matrix  → rows=actual class, cols=predicted class
#                          diagonal values = correct predictions
#   2. Classification Report → Precision, Recall, F1-score per class
# This is the baseline result as required by the teacher's notebook format.


In [ ]:
# train the best model found from CV
best_pipeline = Pipeline([('scaler', StandardScaler()), ('clf', models[best_name])])
best_pipeline.fit(X_train, y_train)
y_pred_best = best_pipeline.predict(X_test)
y_pred_prob = best_pipeline.predict_proba(X_test)[:, 1]

print(f"Best Model ({best_name}) — Test Accuracy: {accuracy_score(y_test, y_pred_best):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=list(class_map.keys())))

# ✅ AFTER RUNNING THIS CELL:
# The best model (identified in Cell 19) is trained and evaluated.
# You will see:
#   Best Model (<Name>) — Test Accuracy: 0.XXXX
#   Classification Report with Precision / Recall / F1 per class
# y_pred_best  → predicted labels (used for confusion matrix)
# y_pred_prob  → predicted probabilities (used for ROC curve)
# Compare this accuracy to the k-NN result above to see the improvement.


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(class_map.keys()))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_name}')
plt.tight_layout()
plt.show()

# ✅ AFTER RUNNING THIS CELL:
# A confusion matrix heatmap is displayed for the best model.
#   - Rows    = Actual class labels
#   - Columns = Predicted class labels
#   - Diagonal cells (dark blue) = correct predictions
#   - Off-diagonal cells = misclassifications
# A perfect model would show all samples on the diagonal only.
# Use this to identify which classes the model confuses with each other.


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='crimson', lw=2, label=f'AUC = {roc_auc:.3f}')
ax.plot([0,1],[0,1], 'k--', lw=1)
ax.fill_between(fpr, tpr, alpha=0.15, color='crimson')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curve — {best_name}')
ax.legend()
plt.tight_layout()
plt.show()
print(f"AUC-ROC: {roc_auc:.4f}")

# ✅ AFTER RUNNING THIS CELL:
# The ROC (Receiver Operating Characteristic) curve is plotted.
# AUC-ROC score is also printed.
# How to interpret:
#   AUC = 1.0  → perfect classifier
#   AUC = 0.5  → random guessing (diagonal dashed line)
#   AUC > 0.9  → excellent
# The shaded area under the curve represents the AUC.
# A higher and more left-leaning curve = better model performance.


In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    estimator=best_pipeline, X=X_train, y=y_train,
    train_sizes=np.linspace(0.1, 1.0, 8), cv=5, scoring='accuracy', n_jobs=-1
)

train_mean = np.mean(train_scores, axis=1)
train_std  = np.std(train_scores,  axis=1)
val_mean   = np.mean(val_scores,   axis=1)
val_std    = np.std(val_scores,    axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, train_mean, 'o-', color='royalblue', lw=2, label='Training accuracy')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='royalblue')
ax.plot(train_sizes, val_mean, 's-', color='darkorange', lw=2, label='Validation accuracy')
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='darkorange')
ax.set_xlabel('Training Set Size')
ax.set_ylabel('Accuracy')
ax.set_title(f'Learning Curve — {best_name}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ✅ AFTER RUNNING THIS CELL:
# A learning curve is plotted for the best model showing:
#   Blue  line = Training accuracy  (usually high, starts near 1.0)
#   Orange line = Validation accuracy (how well it generalises)
# The shaded bands show ± standard deviation across 5 CV folds.
# What to look for:
#   - If both lines converge at a high value → model is well-fitted
#   - Large gap between them → overfitting (model memorising training data)
#   - Both lines low and flat → underfitting (model not learning enough)


In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(StandardScaler().fit_transform(X_resampled))

plot_colors = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12', '#9b59b6', '#1abc9c', '#e67e22', '#34495e']

fig, ax = plt.subplots(figsize=(7, 5))
for (cls, label), col in zip(class_map.items(), plot_colors):
    mask = y_resampled == label
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=col, label=cls.capitalize(), alpha=0.5, s=20, edgecolors='none')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.set_title('PCA Feature Space')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ✅ AFTER RUNNING THIS CELL:
# A 2D PCA scatter plot is displayed showing the feature space.
# Each dot = one image sample, each color = one class.
# PC1 and PC2 are the two principal components (dimensions with most variance).
# How to interpret:
#   - Well-separated color clusters → classes are distinguishable → model will perform well
#   - Overlapping clusters → classes share similar features → harder to classify
# The % variance explained by each axis is shown in the axis labels.


In [ ]:
print(f"{'Model':<24} {'Test Acc':>10}")
print("-" * 36)
for name, clf in models.items():
    p = Pipeline([('s', StandardScaler()), ('c', clf)])
    p.fit(X_train, y_train)
    acc = accuracy_score(y_test, p.predict(X_test))
    marker = " <- BEST" if name == best_name else ""
    print(f"{name:<24} {acc:.4f}{marker}")

# ✅ AFTER RUNNING THIS CELL:
# All 7 models are retrained on X_train and tested on X_test.
# A final summary table is printed:
#   Model                    Test Acc
#   KNN (k=4)                0.XXXX
#   SVM (RBF)                0.XXXX  <- BEST
#   ...
# The best model is marked with '<- BEST'.
# Use this table as the final results summary in your report.
# ✅ Pipeline complete — all cells have been executed successfully!
